In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ClaudeParams:
    model:                str
    max_tokens:           int
    confidence_threshold: float

@dataclass(frozen=True)
class ClaudeValidationConfig:
    root_dir:       Path
    history_path:   Path
    params:         ClaudeParams

In [3]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_claude_config(self) -> ClaudeValidationConfig:
        config = self.config.claude_config
        params = self.params.claude_params
        create_directories([config.root_dir])
        
        return ClaudeValidationConfig(
            root_dir     = Path(config.root_dir),
            history_path = Path(config.root_dir) / "decision_history.json",
            params = ClaudeParams(
                model       = params.model,
                max_tokens  = params.max_tokens,
                confidence_threshold = params.confidence_threshold
            )
        )

In [4]:
import os, sys, json, time, anthropic
from pydantic import BaseModel, Field
from enum import Enum

class ActionType(str, Enum):
    do_nothing           = "do_nothing"
    trigger_retraining   = "trigger_retraining"
    switch_to_lighter    = "switch_to_lighter_model"
    restart_service      = "restart_service"
    scale_out            = "scale_out_service"
    scale_in             = "scale_in_service"
    swap_model           = "swap_model_version"
    
class ExecuteAction(BaseModel):
    action:     ActionType  = Field(description="Action to execute")
    reasoning:  str         = Field(description="Why this action was chosen")


from core.logging import logger
from core.exception import CustomException

class ClaudeValidation:
    def __init__(self, config: ClaudeValidationConfig):
        self.config  = config
        self.client  = self._get_client()
        self.tools   = self._get_tools()
        self.history = self._get_history()
    
    def _get_client(self):
        from dotenv import load_dotenv
        load_dotenv()
        
        try:
            client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
            logger.info(f"Successfully loaded Anthropic client")
            return client
        except Exception as e:
            raise CustomException(e, sys)
        
    def _get_tools(self):
        return [anthropic.types.ToolParam(
            name        = "execute_action",
            description = "Execute a system action when metrics indicate a problem",
            input_schema= ExecuteAction.model_json_schema()
        )]
    
    def _get_history(self) -> list:
        history_path = self.config.history_path
        
        history_path.parent.mkdir(parents=True, exist_ok=True)
        if history_path.exists():
            return json.loads(history_path.read_text())
        return []
    
    def claude_validate(self, metrics: dict, dqn_suggestion: str, dqn_confidence: float) -> dict:
        try:
            response  = self.client.messages.create(
                model = self.config.params.model,
                max_tokens = self.config.params.max_tokens,
                tools = self.tools,
                messages = [{
                    "role": "user",
                    "content": self._build_prompt(metrics, dqn_suggestion, dqn_confidence)
                }]
            )       
        
            for block in response.content:
                if block.type == "tool_use":
                    result = block.input
                    
                    self.history.append({
                        "timestamp":  time.time(),
                        "metrics":    {k: v[0] for k, v in metrics.items()},
                        "dqn":        dqn_suggestion,
                        "claude":     result["action"],
                        "reasoning":  result["reasoning"][:150]
                    })
                    self._save_history()
                    
                    return {
                        "action":    result["action"],
                        "reasoning": result["reasoning"]
                    }

            return {"action": dqn_suggestion, "reasoning": "Claude fallback to DQN"}
        except Exception as e:
            raise CustomException(e, sys)
        
    def _build_prompt(self, metrics: dict, dqn_suggestion: str, dqn_confidence: float) -> str:
        history_str = ""
        if self.history:
            recent = self.history[-5:]
            history_str = f"\nPrevious decisions:\n{json.dumps(recent, indent=2)}\n"
        
        return f"""
            You are an AI system monitor for a medical image segmentation service.
            {history_str}
            Current metrics:
            - CPU:         {metrics.get('current_cpu', [0.0])[0]}%
            - RAM:         {metrics.get('current_ram', [0.0])[0]} MB
            - Latency:     {metrics.get('current_latency', [0.0])[0]}s
            - Drift score: {metrics.get('current_drift', [0.0])[0]} (threshold: {self.config.params.confidence_threshold})

            DQN suggestion: {dqn_suggestion} (confidence: {dqn_confidence:.2f})
            Low confidence means DQN is uncertain — use your judgment.

            Choose the most appropriate action and explain why.
        """.strip()
        
    def _save_history(self):
        self.config.history_path.write_text(
            json.dumps(self.history[-20:], indent=2)
        )

In [5]:
try:
    config = ConfigurationManger()
    claude_config = config.get_claude_config()
    claude_validate = ClaudeValidation(config=claude_config)
    results = claude_validate.claude_validate(
        metrics = {
            "current_cpu":     [0.007],
            "current_ram":     [0.426],
            "current_latency": [0.00475],
            "current_drift":   [0.8]
        },
        dqn_suggestion="trigger_retraining",
        dqn_confidence=0.9 
    )
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 16:18:22,072: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-30 16:18:22,079: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 16:18:22,082: INFO: __init__: created directory at: artifacts]
[2026-04-30 16:18:22,083: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 16:18:22,429: INFO: 3328707342: Successfully loaded Anthropic client]
[2026-04-30 16:18:26,461: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
trigger_retraining
The drift score of 0.8 significantly exceeds the threshold of 0.3, indicating substantial data distribution shift. This is critical for a medical image segmentation service where model accuracy directly impacts patient care. The DQN model has high confidence (0.90) in this recommendation. System resources are abundant (minimal CPU/RAM usage), making this an ideal time to trigger retraining without impacting service performance. Retraining wil

In [6]:
try:
    config = ConfigurationManger()
    claude_config = config.get_claude_config()
    claude_validate = ClaudeValidation(config=claude_config)
    results = claude_validate.claude_validate(
        metrics = {
            "current_cpu":     [0.9],   # high CPU 
            "current_ram":     [0.85],  # high RAM 
            "current_latency": [0.00475],
            "current_drift":   [0.05]   # low drift
        },
        dqn_suggestion="scale_out_service",
        dqn_confidence=0.3
    )
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 16:20:53,745: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-30 16:20:53,757: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 16:20:53,759: INFO: __init__: created directory at: artifacts]
[2026-04-30 16:20:53,760: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 16:20:54,078: INFO: 3328707342: Successfully loaded Anthropic client]
[2026-04-30 16:20:58,033: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
do_nothing
Current system performance is excellent: drift score (0.05) is well below threshold indicating successful recovery from previous retraining; CPU (0.9%) and RAM (0.85 MB) are minimal with no resource constraints; latency (0.00475s) is optimal. DQN's scale_out suggestion has low confidence (0.30), reflecting model uncertainty. Scaling would be wasteful of resources and unnecessary given healthy baseline metrics and stabilized data distribution. Continu